In [ ]:
# ================= NOTEBOOK 3: SHAP & LIME EXPLAINABILITY MODULE =================
!pip install -q shap lime

import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
import shap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
XLMR_DIR = "/content/drive/MyDrive/xlmr_out/checkpoint-2884" # or best model path
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(XLMR_DIR).to(device)
model.eval()

def predict_proba(texts):
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, np.ndarray):
        texts = texts.tolist()
    texts = [str(t) if len(str(t).strip()) > 0 else "." for t in texts]
    
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(texts), 32):
            batch = texts[i : i + 32]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            with torch.amp.autocast("cuda"):
                logits = model(**enc).logits
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

# Filter test hateful posts with valid human rationales
valid_records = []
for idx, row in test[test["y"] == 1].iterrows():
    tokens = [str(t) for t in row["post_tokens"]]
    rats = row["rationales"]
    if isinstance(rats, list) and len(rats) > 0 and len(tokens) >= 3:
        matching = [r for r in rats if isinstance(r, list) and len(r) == len(tokens)]
        if len(matching) > 0:
            avg_mask = np.mean(matching, axis=0)
            human_mask = (avg_mask >= 0.5).astype(int).tolist()
            if sum(human_mask) > 0:
                valid_records.append({"tokens": tokens, "text": " ".join(tokens), "human_mask": human_mask})

print(f"Found {len(valid_records)} valid hateful rationale posts for explainability eval.")